# Bài 4 · Làm quen với pandas

**Lập trình xử lý dữ liệu · 2627-1 · Viện TTNT, UET-VNU**

Chọn **Save a copy in Drive** trước khi sửa.

## Mục tiêu bài học

1. Giải thích vai trò của nhãn hàng, tên cột và dtype.
2. Chọn đúng hàng/cột, phân biệt đọc và gán dữ liệu.
3. Đọc kết quả thống kê cùng đơn vị, giá trị thiếu và mẫu số.

Các bảng nhỏ dùng dữ liệu giả lập. Ví dụ dữ liệu thật dùng Santiago, snapshot 29/06/2026. Môi trường kiểm chứng: pandas 3.0.3; phần sửa dữ liệu áp dụng pandas 3.x.


**Giáo trình:** McKinney, *Python for Data Analysis*, 3rd ed. — §5.1–5.3; đọc/ghi CSV tham khảo §6.1.

**Cập nhật so với sách:** hành vi sửa dữ liệu trong pandas 3 theo [Copy-on-Write](https://pandas.pydata.org/docs/user_guide/copy_on_write.html).

### Ghim pandas 3

Mục 4 của bài dạy đúng hành vi sửa dữ liệu của pandas 3 (cơ chế Copy-on-Write). pandas 2 cho kết quả khác ở những ô đó, nên ô dưới kiểm phiên bản và cài pandas 3 nếu Colab đang dùng bản cũ.


In [ ]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
if int(pd.__version__.split(".")[0]) < 3:
    %pip install -q "pandas>=3"
    print("Đã cài pandas 3. Chọn Restart session and run all rồi chạy lại từ đầu.")

assert int(pd.__version__.split(".")[0]) >= 3, (
    "Notebook này cần pandas 3 nhưng phiên hiện tại vẫn dùng bản cũ. "
    "Chọn Restart session and run all rồi chạy lại từ đầu."
)


### Bài trước

<p>NumPy cho ta shape, dtype, cách chọn phần tử và cách tính trên cả mảng.</p><p>Nhưng mảng chỉ đánh dấu ô theo vị trí: cột thứ 3 là giá hay vĩ độ thì bạn phải tự nhớ.</p><p>Dữ liệu thật có cả số và chữ, nên mỗi hàng và mỗi cột cần một nhãn đi kèm. Đó là việc của pandas.</p>

### Tổng quan

<ol><li>Bảng có nhãn</li><li>Đọc và hiểu dữ liệu</li><li>Chọn hàng, chọn cột</li><li>Tính toán và sửa bảng</li><li>Kiểm tra và lưu</li></ol>

## 1. Bảng có nhãn

Biểu diễn dữ liệu bằng DataFrame và Series.

### Ví dụ về bảng

<p>Dữ liệu giả lập; mỗi hàng là một chỗ ở, giá tính bằng USD/đêm.</p><table class="data-table"><thead><tr><th>chỉ mục</th><th>loại</th><th>giá</th></tr></thead><tbody><tr><td><span style="color:#1E93AB">a</span></td><td>Nguyên căn</td><td>100</td></tr><tr><td><span style="color:#1E93AB">b</span></td><td>Phòng riêng</td><td>50</td></tr><tr><td><span style="color:#1E93AB">c</span></td><td>Nguyên căn</td><td>chưa biết</td></tr></tbody></table>

<p>Ta cần giữ cả <strong>nhãn hàng</strong>, <strong>tên cột</strong> và kiểu dữ liệu của từng cột.</p>

### Tạo bảng với DataFrame

<p><strong>DataFrame</strong> là bảng hai chiều. Mỗi khoá của dict dưới đây tạo một cột.</p>

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "loai": ["Nguyên căn", "Phòng riêng", "Nguyên căn"],
    "gia": [100., 50., np.nan],
}, index=["a", "b", "c"])
print(df)


<p><code>NaN</code> đánh dấu giá trị thiếu; chưa biết giá khác với giá bằng 0.</p>

### Nhãn và kích thước của bảng

<p>Nhãn hàng nằm trong <code>df.index</code>, tên cột nằm trong <code>df.columns</code>. Cả hai đều là đối tượng <strong>Index</strong> — kiểu riêng của pandas dùng để lưu nhãn. Output in ra cả dtype của nhãn.</p>


In [ ]:
print(df.index)


In [ ]:
print(df.columns)


In [ ]:
print(df.shape)


<p>Chỉ mục nằm ngoài các cột dữ liệu, nên bảng có <strong>2 cột</strong>.</p>

### Mỗi cột là một Series

<p>Lấy cột <code>gia</code>: các giá trị vẫn đi cùng nhãn hàng <code>a, b, c</code>.</p>

In [ ]:
gia = df["gia"]
print(gia)


<p><strong>Series</strong> là dữ liệu một chiều có chỉ mục; <code>gia</code> là tên của Series này.</p>

### Tính toán trên Series

<p>Nhân các giá trị với 2, như phép tính trên mảng NumPy:</p>

In [ ]:
print(gia * 2)


<p>Nhãn hàng được giữ lại. Giá trị thiếu vẫn là <code>NaN</code>.</p>

### Mỗi cột có một dtype

<p>Cột loại chứa chữ; cột giá chứa số thực và giá trị thiếu.</p>

In [ ]:
print(df.dtypes)


<p>Hai dòng đầu là dtype của hai cột. Dòng cuối mô tả <strong>Series kết quả</strong> của <code>df.dtypes</code>.</p>

### Bài tập 1: Chọn chỗ ở

Giá giả lập (USD). Phí dọn dẹp chỉ thu **một lần** cho mỗi lượt ở.

| Chỗ ở | Giá mỗi đêm | Phí dọn dẹp |
|---|---:|---:|
| a | 100 | 0 |
| b | 80 | 100 |
| c | 60 | 300 |

1. Tạo DataFrame và tính tổng tiền cho 2 đêm, 7 đêm bằng phép toán trên Series. Mỗi trường hợp chọn chỗ nào rẻ nhất?
2. Từ bao nhiêu đêm thì c rẻ hơn cả a và b? Kiểm tra lại bằng code.


In [ ]:
# Viết dự đoán và lời giải ở đây.


## 2. Đọc và hiểu dữ liệu

Đọc bảng chỗ ở, kiểm tra giá rồi nhận xét mức giá.

### Đọc bảng Airbnb Santiago

<p>Tìm hiểu mức giá chỗ ở tại Santiago từ snapshot ngày 29/06/2026.</p>

In [ ]:
url = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
listings = pd.read_csv(url)
print(listings.shape)


<p><code>read_csv</code> tạo DataFrame: <strong>18.534 hàng, 19 cột</strong>.</p>

### Danh sách các cột

<p>Một hàng là một chỗ ở; mỗi cột giữ một thông tin về chỗ ở đó.</p>

In [ ]:
print(listings.columns)


<p>Đọc tên cột theo vai trò: <code>id</code>, <code>host_id</code> là <strong>khoá</strong> để tra lại bản ghi; <code>price</code>, <code>minimum_nights</code>, <code>number_of_reviews</code> là <strong>số đo</strong>; <code>room_type</code>, <code>neighbourhood</code> là <strong>nhãn phân loại</strong>. Bài này bắt đầu với <code>id</code>, <code>room_type</code> và <code>price</code>.</p>

### Xem trước vài dòng

<p>Mỗi hàng là một chỗ ở; <code>price</code> là giá một đêm, tính bằng peso Chile (CLP).</p>


In [ ]:
print(listings[["id", "room_type", "price"]].head(3))


<p><code>[[...]]</code> chọn các cột theo tên; <code>head(3)</code> lấy 3 hàng đầu.</p><p>Các số 0, 1, 2 bên trái là chỉ mục; <code>id</code> là mã chỗ ở từ file.</p>

### Đếm giá trị thiếu

<p>Thử trên cột giá của bảng nhỏ trước:</p><table class="data-table"><thead><tr><th>Nhãn</th><th><code>gia</code></th><th><code>gia.isna()</code></th></tr></thead><tbody><tr><td>a</td><td>100</td><td>False</td></tr><tr><td>b</td><td>50</td><td>False</td></tr><tr><td>c</td><td>NaN</td><td><strong>True</strong></td></tr></tbody></table>

In [ ]:
print(gia.isna().sum())


<p><code>isna()</code> đánh dấu ô thiếu. <code>sum()</code> đếm các giá trị <code>True</code>.</p>

### Bao nhiêu chỗ ở chưa có giá?



In [ ]:
print(listings["price"].isna().sum())


In [ ]:
print(listings["price"].count())


<p>Trong 18.534 hàng: <strong>846 thiếu giá</strong>, <strong>17.688 có giá</strong>.<br><code>count()</code> chỉ đếm giá trị không thiếu.</p>

### Giá điển hình là bao nhiêu?

<p>Tính trên các giá được ghi nhận, đơn vị CLP/đêm.</p>

In [ ]:
thong_so = ["mean", "median", "max"]
print(listings["price"].agg(thong_so).round())


<p><code>agg</code> chạy các phép tổng hợp đã chọn; <code>round()</code> làm tròn để hiển thị.</p><p>Trung bình cao khoảng gấp đôi trung vị vì vài bản ghi giá rất lớn kéo nó lên. Hai ô dưới đo mức ảnh hưởng đó.</p>

In [ ]:
print(listings["price"].max() / listings["price"].sum())


<p>Riêng bản ghi lớn nhất đã chiếm khoảng 4,6% tổng giá. Bỏ ba giá cao nhất rồi tính lại trung bình:</p>


In [ ]:
ba_gia_cao_nhat = listings.nlargest(3, "price").index
print(listings.drop(ba_gia_cao_nhat)["price"].mean().round())


<p>Trung bình còn khoảng 105.303 CLP, trong khi trung vị gần như không đổi: trung vị chỉ phụ thuộc thứ hạng nên vài giá rất lớn không kéo được nó.</p>


### Bài tập 2: Giá có giảm không?

Giá của ba chỗ ở qua hai lần thu thập (giả lập, USD/đêm).

| Chỗ ở | Lần 1 | Lần 2 |
|---|---:|---:|
| a | 100 | 110 |
| b | 200 | 210 |
| c | 300 | chưa có giá |

1. Tính trung bình các giá đã biết ở mỗi lần thu thập.
2. Tính lại hai trung bình khi chỉ xét a và b ở cả hai lần.
3. Vì sao cách thứ nhất cho thấy giá trung bình giảm, còn cách thứ hai lại tăng?


In [ ]:
# Viết dự đoán và lời giải ở đây.


## 3. Chọn hàng, chọn cột

Chọn theo nhãn, vị trí hoặc điều kiện.


### Chọn một ô


<p>Lấy giá của chỗ ở <code>b</code> trong <code>df</code> đầu bài.</p>


<table class="data-table"><thead><tr><th>Vị trí hàng</th><th>Chỉ mục</th><th>loai</th><th>gia</th></tr></thead><tbody><tr><td>0</td><td>a</td><td>Nguyên căn</td><td>100</td></tr><tr><td>1</td><td>b</td><td>Phòng riêng</td><td>50</td></tr><tr><td>2</td><td>c</td><td>Nguyên căn</td><td>NaN</td></tr></tbody></table>


In [ ]:
print(df.loc["b", "gia"])


In [ ]:
print(df.iloc[1, 1])


<p><code>loc</code> dùng nhãn; <code>iloc</code> dùng vị trí hàng và cột (từ 0).</p>


### Chọn nhiều hàng và cột


<p>Lấy loại và giá của hai chỗ ở <code>b, a</code>, theo đúng thứ tự đó.</p>


In [ ]:
print(df.loc[["b", "a"], ["loai", "gia"]])


<p>Danh sách trước dấu phẩy chọn hàng; danh sách sau chọn cột.</p>


### Giữ kết quả dạng bảng


In [ ]:
print(df.loc[["a", "b"], "gia"])


<p>Tên cột <code>"gia"</code> → Series, shape <code>(2,)</code>.</p>


In [ ]:
print(df.loc[["a", "b"], ["gia"]])


<p>Danh sách <code>["gia"]</code> → DataFrame, shape <code>(2, 1)</code>.</p>


### Chọn các hàng liên tiếp

*📖 Tự học.*


<p>Lấy giá từ hàng <code>a</code> đến hết hàng <code>b</code>:</p>


In [ ]:
print(df.loc["a":"b", "gia"])


<p><code>loc</code> lấy cả nhãn cuối <code>b</code>.</p>


In [ ]:
print(df.iloc[0:2, 1])


<p><code>iloc</code> bỏ vị trí cuối <code>2</code>, như lát cắt NumPy.</p>


### Tạo điều kiện chọn hàng


<p>Chỉ lấy những chỗ ở nguyên căn.</p>


In [ ]:
mask = df["loai"] == "Nguyên căn"
print(mask)


<p>Phép so sánh chạy trên từng hàng và trả về một <strong>Series bool</strong>: cùng chỉ mục <code>a, b, c</code> với <code>df</code>, mỗi hàng một giá trị True hoặc False. <code>Name: loai</code> cho biết mask sinh ra từ cột <code>loai</code>.</p>


### Dùng mask để lấy hàng


In [ ]:
nguyen_can = df.loc[mask, ["loai", "gia"]]
print(nguyen_can)


<p><code>loc</code> giữ các hàng có mask bằng True: <code>a</code> và <code>c</code>.</p>


### Ngoặc vuông và .loc


<p>Đặt tên cột vào ngoặc thì được cột; đặt một Series bool vào ngoặc thì được hàng.</p>


In [ ]:
print(df[mask])


<table class="data-table"><thead><tr><th>Cách viết</th><th>Kết quả</th></tr></thead><tbody><tr><td><code>df["gia"]</code></td><td>Một cột, trả về Series</td></tr><tr><td><code>df[["loai", "gia"]]</code></td><td>Nhiều cột, trả về bảng</td></tr><tr><td><code>df[mask]</code></td><td>Các hàng có True, giữ đủ mọi cột</td></tr><tr><td><code>df.loc[mask, "gia"]</code></td><td>Chọn hàng và cột trong một lệnh</td></tr></tbody></table>


<p>Code bạn đọc trên mạng hoặc do AI sinh ra hay dùng <code>df[mask]</code>. Khi cần nói rõ cả hàng lẫn cột, dùng <code>.loc</code>: hàng trước, cột sau.</p>


### Kết hợp nhiều điều kiện


<p>Trong bảng Santiago, lấy nguyên căn có giá dưới <strong>50.000 CLP/đêm</strong>. Chỉ giữ mã và giá.</p>


In [ ]:
mask = ((listings["room_type"] == "Entire home/apt")
        & (listings["price"] < 50_000))
chon = listings.loc[mask, ["id", "price"]]
print(chon.shape)


<p><code>&amp;</code>: thoả mãn cả hai; <code>|</code>: thoả mãn ít nhất một điều kiện.<br>Đặt mỗi điều kiện trong ngoặc.</p>


### Bài tập 3: Lọc theo ngân sách


<p>Bảng <code>lua_chon</code> (giả lập, USD/đêm). Ngân sách: nguyên căn ≤ 100, phòng riêng ≤ 60.</p>


<table class="data-table"><thead><tr><th>Chỉ mục</th><th>loai</th><th>gia</th></tr></thead><tbody><tr><td>a</td><td>Nguyên căn</td><td>90</td></tr><tr><td>b</td><td>Phòng riêng</td><td>70</td></tr><tr><td>c</td><td>Nguyên căn</td><td>120</td></tr><tr><td>d</td><td>Phòng riêng</td><td>60</td></tr></tbody></table>


<ol><li>Dự đoán các hàng được chọn.</li><li>Viết mask để lấy hai cột <code>loai, gia</code> của các hàng đó.</li></ol>


In [ ]:
lua_chon = pd.DataFrame({
    "loai": ["Nguyên căn", "Phòng riêng", "Nguyên căn", "Phòng riêng"],
    "gia": [90., 70., 120., 60.],
}, index=["a", "b", "c", "d"])


In [ ]:
# Viết lời giải và kiểm tra kết quả ở đây.


## 4. Tính toán và sửa bảng

Tạo bảng giá mới, sửa giá rồi tính lại các khoản tiền.


### Tạo cột tính giá tiền cho ba đêm


<p>Thêm vào <code>df</code> cột tiền ba đêm, tính từ giá mỗi đêm (USD).</p>


In [ ]:
df["ba_dem"] = df["gia"] * 3
print(df[["gia", "ba_dem"]])


<p>Gán vào tên cột mới sẽ thêm cột vào bảng.</p>


### Sửa giá của một chỗ ở


<p>Muốn thử giá mới cho b mà vẫn giữ giá trong <code>df</code>, tạo bản sao rồi sửa:</p>


In [ ]:
bang_gia = df.copy()
bang_gia.loc["b", "gia"] = 60
print(bang_gia["gia"])


<p><code>copy()</code> tạo bảng riêng. Giá của b trong <code>bang_gia</code> là 60; trong <code>df</code> vẫn là 50.</p>


### Sửa Series lấy ra từ bảng


<p>Từ pandas 3, bảng dùng cơ chế <strong>Copy-on-Write</strong>: Series lấy ra chỉ tách khỏi bảng khi bị ghi, nên sửa nó <strong>không làm đổi bảng gốc</strong>.</p>


In [ ]:
cot_gia = bang_gia["gia"]
cot_gia.loc["a"] = 80
print(cot_gia)
print(bang_gia["gia"])


<p>Hai output khác nhau ở hàng <code>a</code>: <code>cot_gia</code> là 80 còn bảng vẫn là 100. Muốn sửa bảng, gán trực tiếp vào <code>bang_gia.loc[hàng, cột]</code>. pandas 2 hành xử khác ở chỗ này; notebook đã ghim pandas 3 ở ô đầu.</p>

Tham khảo: [pandas — Copy-on-Write](https://pandas.pydata.org/docs/user_guide/copy_on_write.html).


### Hai lần ngoặc thì bảng không đổi


<p>Dạng viết này gặp nhiều trong code cũ và trong code do AI sinh ra:</p>


In [ ]:
bang_gia["gia"]["a"] = 80


In [ ]:
print(bang_gia.loc["a", "gia"])


<p>pandas in cảnh báo <code>ChainedAssignmentError</code> và bảng không đổi: giá của <code>a</code> vẫn là 100. Ngoặc thứ nhất lấy ra một Series, ngoặc thứ hai ghi vào Series đó. Viết một lệnh duy nhất: <code>bang_gia.loc["a", "gia"] = 80</code>.</p>

Cảnh báo này là một <code>Warning</code> chứ không phải lỗi dừng chương trình: code vẫn chạy tiếp, nên dễ bị bỏ qua.


### Giảm giá cho chỗ ở nguyên căn


<p>Giảm 10% giá nguyên căn trong <code>bang_gia</code>; giữ giá phòng riêng.</p>


In [ ]:
mask = bang_gia["loai"] == "Nguyên căn"
bang_gia.loc[mask, "gia"] = bang_gia.loc[mask, "gia"] * 0.9
print(bang_gia[["loai", "gia"]])


### Tính lại tiền sau khi sửa giá


In [ ]:
print(bang_gia[["gia", "ba_dem"]])


<p><code>ba_dem</code> vẫn chứa kết quả cũ. Cần chạy lại phép tính:</p>


In [ ]:
bang_gia["ba_dem"] = bang_gia["gia"] * 3
print(bang_gia[["gia", "ba_dem"]])


<p>Một cột lưu <strong>giá trị đã tính</strong>, không lưu công thức như ô Excel: sửa giá xong, cột phụ thuộc vẫn giữ số cũ cho tới khi bạn tính lại.</p>


### Bài tập 4: Báo giá ba đêm


<p>Dùng <code>df</code> đầu bài. Giảm 10% giá nguyên căn, rồi tính tiền ba đêm. Giữ nguyên <code>df</code>.</p>


```python
bao_gia = df.copy()
bao_gia["ba_dem"] = bao_gia["gia"] * 3
mask = bao_gia["loai"] == "Nguyên căn"
gia_nguyen_can = bao_gia.loc[mask, "gia"]
gia_nguyen_can *= 0.9
```


<ol><li>Code trên cho tiền ba đêm của a là bao nhiêu? Theo yêu cầu, phải là bao nhiêu?</li><li>Sửa code và kiểm tra cả ba hàng.</li></ol>


In [ ]:
# Viết lời giải và kiểm tra kết quả ở đây.


## 5. Kiểm tra và lưu

Kiểm tra các giá trị cần chú ý và thông tin được giữ trong file.


### Xem những chỗ ở giá cao nhất


<p>Ở mục 2, giá lớn nhất gần <strong>97 triệu CLP/đêm</strong>. Xem các hàng có giá cao nhất:</p>


In [ ]:
print(listings.nlargest(3, "price")[["id", "price"]])


<p><code>nlargest(3, "price")</code> lấy ba hàng có giá cao nhất. Dùng <code>id</code> để tra lại chỗ ở trong nguồn dữ liệu.</p>


### Tỷ lệ giá dưới 80 USD/đêm


<p>Với Series bool, <code>mean()</code> là tỷ lệ giá trị True.</p>


<table class="data-table"><thead><tr><th>Chỉ mục</th><th>gia</th><th>gia &lt; 80</th></tr></thead><tbody><tr><td>a</td><td>100</td><td>False</td></tr><tr><td>b</td><td>50</td><td>True</td></tr><tr><td>c</td><td>NaN</td><td>False</td></tr></tbody></table>


In [ ]:
print((df["gia"] < 80).mean())


<p>Trên cả bảng: 1/3. Chỉ xét hàng có giá bằng <code>notna()</code>:</p>


In [ ]:
co_gia = df["gia"].notna()
print((df.loc[co_gia, "gia"] < 80).mean())


### Cùng câu hỏi trên bảng Santiago

<p>Tỷ lệ chỗ ở có giá dưới 50.000 CLP/đêm, tính trên toàn bảng:</p>


In [ ]:
print((listings["price"] < 50_000).mean())


<p>Rồi chỉ tính trên 17.688 hàng có giá:</p>


In [ ]:
co_gia = listings["price"].notna()
print((listings.loc[co_gia, "price"] < 50_000).mean())


<p>Mẫu số đổi thì tỷ lệ đổi: 36,0% và 37,7% trả lời hai câu hỏi khác nhau. Xác định tập hàng trước, rồi mới tính.</p>


### Lưu bảng giá để dùng tiếp


<p>Chỉ giữ các hàng có giá và ba cột <code>id, room_type, price</code>.</p>


In [ ]:
cot = ["id", "room_type", "price"]
co_gia = listings["price"].notna()
ket_qua = listings.loc[co_gia, cot]
ket_qua.to_csv("gia_co_thong_tin.csv", index=False)


<p><code>index=False</code> bỏ chỉ mục 0, 1, … khi ghi file.<br>Mã chỗ ở vẫn được giữ trong cột <code>id</code>.</p><p>Đây chưa phải bảng đã kiểm tra xong chất lượng: các giá rất lớn vẫn còn đó. Lưu lại để làm tiếp.</p>


### Đọc lại file vừa lưu


In [ ]:
doc_lai = pd.read_csv("gia_co_thong_tin.csv")
print(doc_lai.shape)


In [ ]:
print(doc_lai["price"].isna().sum())


In [ ]:
print(doc_lai.columns)


In [ ]:
print(doc_lai["price"].sum() == ket_qua["price"].sum())


<p>Đối chiếu: <strong>17.688 hàng, 3 cột, không thiếu giá</strong>, tên cột còn đủ và tổng giá đọc lại bằng đúng tổng giá trước khi ghi.</p>


### Bài tập 5: Giữ mã chỗ ở trong file


<p>Bảng <code>df</code> đầu bài dùng <code>a, b, c</code> làm chỉ mục. Một bạn gửi bảng giá bằng cách:</p>


```python
bang_gui = df[["gia"]]
bang_gui.to_csv("bang_gui.csv", index=False)
```


<ol><li>Người nhận chỉ có file CSV. Họ có biết mỗi giá thuộc chỗ ở nào không?</li><li>Sửa cách lưu để giữ cả mã chỗ ở và giá. Đọc lại file để kiểm tra.</li></ol>


In [ ]:
# Viết lời giải và kiểm tra kết quả ở đây.


## Tự học theo giáo trình, chương 5

Các ví dụ sau mở rộng phần thao tác cơ bản (§5.2) và thống kê (§5.3). Không cần học thuộc danh sách hàm.

### Mỗi loại chỗ ở có bao nhiêu dòng?

Đếm trên snapshot, không suy rộng thành toàn bộ thị trường.

In [ ]:
print(listings["room_type"].value_counts())


### Chọn một trong nhiều loại

Lấy phòng riêng hoặc phòng chung, rồi sắp giá từ thấp đến cao.

In [ ]:
chon_loai = listings["room_type"].isin(["Private room", "Shared room"])
print(listings.loc[chon_loai, ["room_type", "price"]].sort_values("price").head())


### Kiểm tra toàn bảng

Đọc info và describe để xác định cột cần xem kỹ; các thống kê không tự chứng minh dữ liệu đúng.

In [ ]:
listings.info()
print(listings["price"].describe())


### Cột nhãn phân loại và dtype `category`

Cột `neighbourhood` chỉ có 31 giá trị khác nhau nhưng được lưu thành 18.534 chuỗi. Đo dung lượng trước và sau khi đổi sang dtype `category`.


In [ ]:
print(listings["neighbourhood"].nunique())


In [ ]:
print(listings["neighbourhood"].memory_usage(deep=True))


In [ ]:
print(listings["neighbourhood"].astype("category").memory_usage(deep=True))


Từ 1.140.968 byte xuống 20.646 byte, nhỏ hơn khoảng 55 lần: `category` lưu 31 nhãn một lần rồi để mỗi hàng giữ một mã số.

Dùng `category` khi cột là nhãn phân loại lặp lại nhiều lần (`room_type`, `neighbourhood`). Không dùng cho cột khoá như `id` — mỗi hàng một giá trị khác nhau nên chẳng tiết kiệm được gì.


### Làm việc với AI thì sao?


<div class="slide-badge-row"><span class="ai-badge">🤖 Làm việc với AI</span></div><div class="good-box"><span class="box-title">AI làm tốt</span>Viết biểu thức lọc, tính cột mới và xuất file khi yêu cầu rõ.</div><div class="warn-box"><span class="box-title">AI hay sai</span>Nhầm nhãn với vị trí; sửa Series thay vì bảng; quên tính lại cột hoặc làm mất mã khi lưu.</div>


### Kiểm chứng thế nào?


<ol><li>Dự đoán các hàng được chọn và tính tay vài kết quả.</li><li>Sau phép gán, xem bảng cần sửa và các cột phụ thuộc.</li><li>Đọc lại file: mã chỗ ở có còn đi cùng đúng giá không?</li></ol>


### Đọc thêm


<ol><li><a href="https://wesmckinney.com/book/pandas-basics">McKinney, chương 5: Series, DataFrame và thao tác cơ bản</a></li><li><a href="https://pandas.pydata.org/docs/user_guide/copy_on_write.html">pandas 3: Copy-on-Write khi sửa dữ liệu</a></li><li><a href="https://wesmckinney.com/book/accessing-data">McKinney, §6.1: đọc và ghi CSV</a></li></ol>


---

## Tóm tắt bài học

- DataFrame giữ nhãn hàng và tên cột; mỗi cột là một Series có dtype riêng.
- Đọc dữ liệu xong, xem ngay kích thước, tên cột và số giá trị thiếu.
- Dùng nhãn, vị trí hoặc điều kiện để chọn hàng và cột.
- Muốn sửa bảng nào, gán trực tiếp vào bảng đó.
- Tính lại các cột phụ thuộc sau khi sửa dữ liệu.
- Kiểm tra tập hàng được tính và mã được giữ trong file.

Chạy **Restart & Run all** để kiểm tra từ đầu.

**Bài sau:** khớp nhãn, tính theo nhóm và ghép bảng.
